# 01 — Residue Classes mod 6

Purpose: first measurable constraint result.

\[
p > 3 \Rightarrow p \equiv \pm 1 \pmod{6}
\]

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, math, zipfile, json
import matplotlib.pyplot as plt

NOTEBOOK_ID="01_residue_classes_mod6"
NOTEBOOK_NUM=NOTEBOOK_ID.split("_")[0]

OUT=Path(NOTEBOOK_ID)
DATA_DIR=OUT/"data"
DOCS_DIR=OUT/"docs"
FIG_DIR=OUT/"figures"
TEX_DIR=OUT/"tex"

for d in [DATA_DIR,DOCS_DIR,FIG_DIR,TEX_DIR]:
    d.mkdir(parents=True,exist_ok=True)

In [ ]:
def sieve(n):
    s=np.ones(n,dtype=bool)
    s[:2]=False
    for i in range(2,int(math.sqrt(n))+1):
        if s[i]:
            s[i*i:n:i]=False
    return np.nonzero(s)[0]

N_MAX=200000
primes=sieve(N_MAX)
p=primes[primes>3]
ints=np.arange(1,N_MAX)

In [ ]:
def counts(v):
    c=np.bincount(v%6,minlength=6)
    return pd.DataFrame({"r":range(6),"count":c,"share":c/c.sum()})

int_df=counts(ints)
prime_df=counts(p)
df=int_df.merge(prime_df,on="r",suffixes=("_int","_prime"))
df

In [ ]:
valid=np.isin(p%6,[1,5])
cgcs=valid.mean()
drift=1-cgcs

measurement={"cgcs":float(cgcs),"drift":float(drift)}
measurement

In [ ]:
# figure 1
fig,ax=plt.subplots()
ax.bar(df["r"]-0.2,df["share_int"],0.4,label="ints")
ax.bar(df["r"]+0.2,df["share_prime"],0.4,label="primes")
ax.legend()

fig1=FIG_DIR/f"{NOTEBOOK_NUM}_residue.png"
fig.savefig(fig1)
plt.close(fig)

# figure 2
scales=[10,100,1000,10000,100000]
rows=[]
for s in scales:
    ps=primes[(primes>3)&(primes<s)]
    if len(ps)==0: continue
    share=np.isin(ps%6,[1,5]).mean()
    rows.append({"x":s,"cgcs":share,"drift":1-share})
scale_df=pd.DataFrame(rows)

fig,ax=plt.subplots()
ax.plot(scale_df["x"],scale_df["cgcs"],label="cgcs")
ax.plot(scale_df["x"],scale_df["drift"],label="drift")
ax.set_xscale("log")
ax.legend()

fig2=FIG_DIR/f"{NOTEBOOK_NUM}_scale.png"
fig.savefig(fig2)
plt.close(fig)

fig1,fig2

In [ ]:
# interpretation + figures auto section
interpretation=f"CGCS={cgcs:.3f}, drift={drift:.3f}"

figs=[fig1,fig2]
fig_md="\n\n## Figures\n\n"
for i,f in enumerate(figs,1):
    fig_md+=f"### Figure {i}\n![fig](../figures/{f.name})\n\n"

interp_path=DOCS_DIR/f"{NOTEBOOK_NUM}_interpretation.md"
interp_path.write_text(interpretation+fig_md)

In [ ]:
# export zip
EXPORT_NAME=f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR,DATA_DIR,FIG_DIR,TEX_DIR]:
        for pth in folder.rglob("*"):
            if pth.is_file():
                z.write(pth,pth.as_posix())

print(EXPORT_NAME)